# Classification NBA Model

## Configuration

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training
)
from sklearn.model_selection import KFold, cross_validate, train_test_split
from xgboost import XGBClassifier


## Load Data

In [ ]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260310.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [3]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=50, max_na_per_row=6, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 11170 rows
Basic cleaning complete: 8561 rows remaining

Starting advanced column cleaning with 2978 columns

Advanced column cleaning complete: 2978 → 1980 columns (998 removed)


Dropping NA rows for SEASON_YEAR 2017...
   Removed 0 rows with NaN values from 2017 season

Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 0 (0.0%)
  Critical columns requiring data: 4
  Columns zero-filled: 98
  Infer pairs applied: 20/136
  Remaining NaN cells: 944087

Dropping rows with more than 6 NaN values...
Removed 3922 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (4639, 1980)


In [4]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_to_train.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1789,total_consensus_pct_under,72,1.55,2024.0
1790,spread_consensus_pct_home,67,1.44,2024.0
1749,total_consensus_pct_over_TREND_SLOPE_LAST_5_HO...,15,0.32,2023.0
1755,spread_consensus_pct_home_TREND_SLOPE_LAST_5_H...,15,0.32,2023.0
1751,total_consensus_pct_under_TREND_SLOPE_LAST_5_H...,14,0.30,2023.0
1796,ml_consensus_opener_price_away,14,0.30,2025.0
1797,ml_consensus_opener_price_home,14,0.30,2025.0
1430,total_betmgm_price_under_LAST_HOME_AWAY_5_MATC...,12,0.26,2021.0
1427,total_betmgm_price_over_LAST_HOME_AWAY_5_MATCH...,12,0.26,2021.0
1457,spread_betmgm_price_LAST_HOME_AWAY_5_MATCHES_D...,12,0.26,2021.0


In [5]:
BET365_LINE_COL =  "TOTAL_LINE_bet365"
# BET365_LINE_COL =  "total_bet365_line_over"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [6]:
# df_to_train['LINE_ERROR'] = df_to_train['TOTAL_POINTS'] - df_to_train['TOTAL_OVER_UNDER_LINE']
df_to_train['LINE_ERROR'] = df_to_train['TOTAL_POINTS'] - df_to_train[BET365_LINE_COL]

In [7]:
df_to_train['GAME_DATE'] = pd.to_datetime(df_to_train['GAME_DATE'])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

## Train / Test

In [8]:
X = df_to_train.drop(['TOTAL_POINTS', 'LINE_ERROR', 'SEASON_YEAR', 'GAME_DATE'], axis=1, errors='ignore')
y = df_to_train['LINE_ERROR']

In [9]:
# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=16, shuffle=False)

In [10]:
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
# Check number of coulmns
print(f"Number of columns in training set: {X_train.shape[1]}")
print(f"Number of columns in test set: {X_test.shape[1]}")

Training set size: 3943
Test set size: 696
Number of columns in training set: 1977
Number of columns in test set: 1977


## Cross-validation

In [ ]:
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer, root_mean_squared_error
from sklearn.dummy import DummyRegressor


In [12]:
# Declare KFold
kf = TimeSeriesSplit(n_splits=5)

In [13]:
def over_under_betting_accuracy(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    true_sign = np.sign(y_true)
    pred_sign = np.sign(y_pred)

    valid = (true_sign != 0) & (pred_sign != 0)
    if not np.any(valid):
        return 0.0

    return float(np.mean(true_sign[valid] == pred_sign[valid]))


class OverUnderScorer:
    def __call__(self, estimator, X, y_true):
        y_pred = estimator.predict(X)
        return over_under_betting_accuracy(y_true, y_pred)


over_under_scorer = OverUnderScorer()

In [14]:
# Declare scores to be used
scoring = {
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(root_mean_squared_error),
    'MAE': make_scorer(mean_absolute_error),
    'OU_Betting_Accuracy': over_under_scorer,
}

In [15]:
def print_metrics(cv_results):
    for sc in scoring.keys():
        if sc == 'OU_Betting_Accuracy':
            print(f'Train {sc}:', f"{cv_results[f'train_{sc}'].mean():.2%}")
            print(f'Validation {sc}:', f"{cv_results[f'test_{sc}'].mean():.2%}")
        else:
            print(f'Train {sc}:', cv_results[f'train_{sc}'].mean().round(5))
            print(f'Validation {sc}:', cv_results[f'test_{sc}'].mean().round(5))
        print()

In [16]:
def real_vs_pred(model, X_train, y_train):
    preds = cross_val_predict(model, X_train, y_train, cv=kf, n_jobs=-1)
    x_line = np.arange(y_train.min(), y_train.max())
    plt.scatter(y_train, preds)
    plt.plot(x_line, x_line, color='orange')
    plt.xlabel('Real target')
    plt.ylabel('Predicted target')
    plt.show()

## Baseline

In [17]:
from sklearn.dummy import DummyRegressor

season_bl = DummyRegressor(strategy='mean')
cv_results = cross_validate(season_bl, X_train, y_train, cv=kf,
                            scoring=scoring, return_train_score=True)
season_bl.fit(X_train, y_train)
print_metrics(cv_results)

Train MSE: 296.21817
Validation MSE: 290.53851

Train RMSE: 17.21072
Validation RMSE: 17.03947

Train MAE: 13.60096
Validation MAE: 13.58518

Train OU_Betting_Accuracy: 52.10%
Validation OU_Betting_Accuracy: 50.74%



In [18]:
# Baseline 3: Predict the betting line (TOTAL_OVER_UNDER_LINE)
y_pred_baseline_3 = X_train[BET365_LINE_COL]

# Evaluate
mse = mean_squared_error(y_train, y_pred_baseline_3)
mae = mean_absolute_error(y_train, y_pred_baseline_3)
rmse = root_mean_squared_error(y_train, y_pred_baseline_3)
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")

MSE: 51250.27
RMSE: 226.39
MAE: 225.55


## Logistic Regression

In [19]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
cv_results = cross_validate(lr, X_train.fillna(0), y_train, cv=kf,
                            scoring=scoring, return_train_score=True)

print_metrics(cv_results)

Train MSE: 79.20381
Validation MSE: 25282357.36633

Train RMSE: 7.1969
Validation RMSE: 3727.72567

Train MAE: 5.51737
Validation MAE: 937.21253

Train OU_Betting_Accuracy: 85.79%
Validation OU_Betting_Accuracy: 49.75%



In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

# Example XGBoost regressor:
xgb_reg = XGBRegressor(
    max_depth=4,
    learning_rate=0.02,
    n_estimators=250,
    subsample=0.5,       # Equivalent to max_samples in GBRegressor
    colsample_bytree=0.6, # Equivalent to max_features in GBRegressor
    n_jobs=-2,
    random_state=16)

cv_results = cross_validate(
    xgb_reg, 
    X_train, y_train, 
    cv=kf, 
    scoring=scoring,      # Use your custom scoring or e.g. 'neg_mean_absolute_error'
    return_train_score=True,
    n_jobs=-2
)
# Print metrics
print_metrics(cv_results)
# Train final model on full train set
xgb_reg.fit(X_train, y_train)


Train MSE: 126.79329
Validation MSE: 293.73239

Train RMSE: 11.01486
Validation RMSE: 17.13091

Train MAE: 8.70743
Validation MAE: 13.64793

Train OU_Betting_Accuracy: 86.98%
Validation OU_Betting_Accuracy: 52.62%



XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.02, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=4, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=250, n_jobs=-2,
             num_parallel_tree=None, random_state=16, ...)

In [21]:
feature_importances = xgb_reg.feature_importances_
important_features = np.argsort(feature_importances)[::-1]  
feature_importances = pd.DataFrame({
    'Feature': X_train.columns[important_features],
    'Importance': feature_importances[important_features]
}).sort_values(by="Importance", ascending=False)
feature_importances

,Feature,Importance
0,TOTAL_LINE_caesars,0.001635
1,DIFF_FROM_LINE_caesars_SEASON_BEFORE_STD_TREND...,0.001590
2,TOTAL_INJURED_PLAYER_MIN_BEFORE_TEAM_AWAY,0.001586
3,DIFF_FROM_LINE_fanduel_LAST_ALL_1_MATCHES_BEFO...,0.001469
4,DIFF_FROM_LINE_consensus_opener_LAST_ALL_3_MAT...,0.001463
...,...,...
1602,odds_spread_home_is_fav_draftkings,0.000000
1603,total_bet365_price_over_SEASON_BEFORE_AVG_TEAM...,0.000000
1604,total_bet365_price_under_LAST_ALL_5_MATCHES_BE...,0.000000
1605,total_bet365_price_under_LAST_HOME_AWAY_5_MATC...,0.000000


In [23]:
import numpy as np
import pandas as pd


def error_sign_accuracy(y_true_error, y_pred_error) -> float:
    """
    Same sign of (predicted error) and (true error).

    Push handling:
    - If either sign is 0, exclude that sample from scoring.
      (If you prefer to count (0,0) as correct, I can change it.)
    """
    y_true_error = np.asarray(y_true_error, dtype=float)
    y_pred_error = np.asarray(y_pred_error, dtype=float)

    true_sign = np.sign(y_true_error)
    pred_sign = np.sign(y_pred_error)

    valid = (true_sign != 0) & (pred_sign != 0)
    if not np.any(valid):
        return np.nan

    return float(np.mean(true_sign[valid] == pred_sign[valid]))


def evaluate_error_thresholds(
    model,
    X_test: pd.DataFrame,
    y_test_error: pd.Series,
    thresholds=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10),
):
    """
    Evaluate directional accuracy at different confidence thresholds.

    Threshold rule:
    - Include a game if abs(predicted_error) > t
    """
    # Predict error directly
    y_pred_error = np.asarray(model.predict(X_test), dtype=float)

    margin = np.abs(y_pred_error)

    rows = []
    n_total = len(y_test_error)

    y_true_error_np = y_test_error.to_numpy(dtype=float)

    for t in thresholds:
        mask = margin > t
        n = int(mask.sum())

        acc = (
            np.nan
            if n == 0
            else error_sign_accuracy(y_true_error_np[mask], y_pred_error[mask])
        )

        rows.append(
            {
                "threshold_abs_pred_error_gt": t,
                "n_games": n,
                "pct_of_test": (n / n_total) if n_total else np.nan,
                "directional_accuracy": acc,
            }
        )

    return pd.DataFrame(rows), y_pred_error
results_df, y_pred_test_error = evaluate_error_thresholds(
    model=xgb_reg,
    X_test=X_test,
    y_test_error=y_test,      # y_test must be the REAL ERROR series
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
    )
)



,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,696,100.0%,51.59%
1,1,448,64.4%,52.49%
2,2,233,33.5%,50.44%
3,3,98,14.1%,49.47%
4,4,48,6.9%,53.19%
5,5,16,2.3%,43.75%
6,6,2,0.3%,50.00%
7,7,0,0.0%,nan%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


# Optuna

In [37]:
import numpy as np
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from xgboost import XGBRegressor
from sklearn.model_selection import KFold



def over_under_betting_accuracy(true_error: np.ndarray, pred_error: np.ndarray) -> float:
    true_sign = np.sign(true_error)
    pred_sign = np.sign(pred_error)

    valid = (true_sign != 0) & (pred_sign != 0)
    if not np.any(valid):
        return np.nan

    return float(np.mean(true_sign[valid] == pred_sign[valid]))


def ou_accuracy_with_threshold_errors(
    true_error: np.ndarray,
    pred_error: np.ndarray,
    threshold_abs: float = 0.0,
    min_coverage: float = 0.25,
):
    # "Edge" in points relative to the line
    margin = np.abs(pred_error)

    # Bet only when predicted edge exceeds threshold
    mask = margin > threshold_abs
    coverage = float(np.mean(mask))

    if coverage < min_coverage or mask.sum() == 0:
        return 0.0, coverage

    score = over_under_betting_accuracy(true_error[mask], pred_error[mask])

    # If everything got filtered as pushes (rare, but possible), penalize
    if np.isnan(score):
        return 0.0, coverage

    return score, coverage

def _predict_best(model, X):
    # Use best iteration if early stopping happened
    if getattr(model, "best_iteration", None) is not None:
        # Newer XGBoost
        try:
            return model.predict(X, iteration_range=(0, model.best_iteration + 1))
        except TypeError:
            # Older compatibility path
            ntree_limit = getattr(model, "best_ntree_limit", None)
            if ntree_limit is not None:
                return model.predict(X, ntree_limit=ntree_limit)
    return model.predict(X)

def objective(trial, X, y):
    threshold_abs = trial.suggest_float("threshold_abs", 0.0, 10.0, step=0.5)
    min_coverage = 0.25

    params = {
        "booster": "gbtree",
        "tree_method": "hist",
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "min_child_weight": trial.suggest_float("min_child_weight", 5.0, 20.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.50, 0.80),

        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.30, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 20.0, log=True),

        "n_estimators": trial.suggest_int("n_estimators", 225, 550, step=25),
        "eval_metric": "rmse",
        "early_stopping_rounds": 200,

        "random_state": 16,
        "n_jobs": -1,
    }

    fold_scores = []
    fold_coverages = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr = y.iloc[tr_idx].to_numpy()
        y_va = y.iloc[va_idx].to_numpy()

        model = XGBRegressor(**params)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
        )

        y_pred_error = _predict_best(model, X_va)

        score, coverage = ou_accuracy_with_threshold_errors(
            true_error=y_va,
            pred_error=y_pred_error,
            threshold_abs=threshold_abs,
            min_coverage=min_coverage,
        )

        fold_scores.append(score)
        fold_coverages.append(coverage)

        trial.report(float(np.mean(fold_scores)), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    trial.set_user_attr("mean_coverage", float(np.mean(fold_coverages)))
    return float(np.mean(fold_scores))

# ----------------------------
# Run the search (2 to 3 hours)
# ----------------------------
# Make sure X_train includes TOTAL_OVER_UNDER_LINE, since you use it in the metric.
# X_train, y_train are your current train split.
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=16),
    pruner=MedianPruner(n_warmup_steps=2),
)

# Time budget: 3 hours. You can adjust to 2*3600 if you want.
study.optimize(lambda t: objective(t, X_train, y_train), timeout= 2.5*3600, n_jobs=1)

print("Best value (CV OU success):", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(k, v)

# ----------------------------
# Train final model on full train set
# ----------------------------
best_params = study.best_params.copy()
best_threshold = best_params.pop("threshold_abs")  # strategy threshold

final_params = {
    "booster": "gbtree",
    "tree_method": "hist",
    "random_state": 16,
    "n_jobs": -1,
    **best_params,
}

final_model = XGBRegressor(**final_params)

[I 2026-03-07 14:27:19,253] A new study created in memory with name: no-name-d99ae18a-a461-497b-9e1c-0be203909938


[I 2026-03-07 14:29:23,271] Trial 0 finished with value: 0.21607702794819356 and parameters: {'threshold_abs': 2.0, 'max_depth': 4, 'min_child_weight': 10.728161890750807, 'gamma': 0.22800975066403162, 'subsample': 0.6164373012086636, 'colsample_bytree': 0.5669242825073867, 'learning_rate': 0.05082275623350138, 'reg_alpha': 0.004517786465395959, 'reg_lambda': 0.01706650116663653, 'n_estimators': 550}. Best is trial 0 with value: 0.21607702794819356.
[I 2026-03-07 14:31:04,554] Trial 1 finished with value: 0.0 and parameters: {'threshold_abs': 5.5, 'max_depth': 2, 'min_child_weight': 13.615793124046794, 'gamma': 0.7922608676158321, 'subsample': 0.550168784046414, 'colsample_bytree': 0.5880461768306593, 'learning_rate': 0.05316051828678, 'reg_alpha': 0.07195423366266383, 'reg_lambda': 0.05127746977873438, 'n_estimators': 375}. Best is trial 0 with value: 0.21607702794819356.
[I 2026-03-07 14:34:18,947] Trial 2 finished with value: 0.308021425515007 and parameters: {'threshold_abs': 1.0, 

Best value (CV OU success): 0.5422554105505287
Best params:
threshold_abs 0.5
max_depth 4
min_child_weight 5.564329233342414
gamma 3.1840252113331995
subsample 0.7219803411995012
colsample_bytree 0.6385727252748334
learning_rate 0.03933797911890074
reg_alpha 5.120272747276847
reg_lambda 0.09591278576208848
n_estimators 250


## Check in Test set

In [38]:
final_model.fit(X_train, y_train, verbose=False)
results_df_final, y_pred_test = evaluate_error_thresholds(
    model=final_model,
    X_test=X_test,
    y_test_error=y_test,
    thresholds=range(0, 11)  # 0..10
)

# Pretty display
display(results_df_final.style.format({
    "pct_of_test": "{:.1%}",
    "error_betting_accuracy": "{:.2%}",
}))


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,691,100.0%,0.505109
1,1,531,76.8%,0.516190
2,2,385,55.7%,0.482850
3,3,256,37.0%,0.524000
4,4,166,24.0%,0.518293
5,5,103,14.9%,0.524272
6,6,51,7.4%,0.450980
7,7,22,3.2%,0.545455
8,8,10,1.4%,0.700000
9,9,6,0.9%,0.500000


In [39]:
feature_importances = final_model.feature_importances_
important_features = np.argsort(feature_importances)[::-1]  
feature_importances = pd.DataFrame({
    'Feature': X_train.columns[important_features],
    'Importance': feature_importances[important_features]
}).sort_values(by="Importance", ascending=False)
feature_importances

,Feature,Importance
0,TOTAL_LINE_betmgm_LAST_5_WMA_BEFORE_TEAM_AWAY,0.002099
1,odds_implied_points_gap,0.001816
2,TOP5_INJURED_STREAK_PTS_BEFORE_TEAM_AWAY,0.001745
3,TOTAL_TOP_INJURED_PLAYER_MIN_BEFORE_TEAM_AWAY,0.001740
4,DIFF_FROM_LINE_caesars_TREND_SLOPE_LAST_5_GAME...,0.001682
...,...,...
1631,TOTAL_LINE_bet365_LAST_ALL_5_MATCHES_BEFORE_TE...,0.000000
1632,DIFF_FROM_LINE_betmgm_LAST_ALL_2_MATCHES_BEFOR...,0.000000
1633,DIFF_FROM_LINE_betmgm_LAST_HOME_AWAY_5_MATCHES...,0.000000
1634,DIFF_FROM_LINE_betmgm_LAST_ALL_5_MATCHES_BEFOR...,0.000000


In [40]:

import json
from pathlib import Path

import joblib
import numpy as np
from xgboost import XGBRegressor


def train_final_model(
    X_all,
    y_all,
    study,
    *,
    use_early_stopping: bool = True,
    val_frac: float = 0.1,
    random_state: int = 16,
):
    """
    Train a final XGBRegressor using Optuna best params.
    Also returns the betting threshold (threshold_abs) stored in the study params.
    """
    best_params = study.best_params.copy()
    best_threshold = float(best_params.pop("threshold_abs"))

    final_params = {
        "booster": "gbtree",
        "tree_method": "hist",
        "random_state": random_state,
        "n_jobs": -1,
        "eval_metric": "rmse",
        **best_params,
    }

    model = XGBRegressor(**final_params)

    if not use_early_stopping:
        # Train on all data without early stopping
        model.fit(X_all, y_all, verbose=False)
        return model, best_threshold

    # Keep early stopping by holding out a small validation split
    n = len(X_all)
    rng = np.random.default_rng(random_state)
    idx = np.arange(n)
    rng.shuffle(idx)

    n_val = max(1, int(round(val_frac * n)))
    val_idx = idx[:n_val]
    tr_idx = idx[n_val:]

    X_tr, y_tr = X_all.iloc[tr_idx], y_all.iloc[tr_idx]
    X_va, y_va = X_all.iloc[val_idx], y_all.iloc[val_idx]

    model.set_params(early_stopping_rounds=200)
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False,
    )
    return model, best_threshold


def save_model_bundle(
    model: XGBRegressor,
    threshold_abs: float,
    feature_names: list[str],
    out_dir: str | Path,
    name: str = "xgb_ou_model",
):
    """
    Saves:
      - model.joblib (sklearn wrapper)
      - metadata.json (threshold + feature order)
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    model_path = out_dir / f"{name}.joblib"
    meta_path = out_dir / f"{name}.meta.json"

    joblib.dump(model, model_path)

    metadata = {
        "threshold_abs": float(threshold_abs),
        "feature_names": list(feature_names),
        "xgboost_version_note": "Saved via joblib XGBRegressor wrapper",
    }
    meta_path.write_text(json.dumps(metadata, indent=2))

    return model_path, meta_path


def load_model_bundle(model_path: str | Path, meta_path: str | Path):
    model = joblib.load(model_path)
    metadata = json.loads(Path(meta_path).read_text())
    return model, metadata


# -------------------------
# Example usage
# -------------------------

# 1) Build ALL data (replace with your variables)
X_all = X  # your full feature DF, must include TOTAL_OVER_UNDER_LINE column for later scoring/decision rule
y_all = y  # your full target Series

# 2) Train final model
final_model, best_threshold = train_final_model(
    X_all=X_all,
    y_all=y_all,
    study=study,
    use_early_stopping=True,   # or False if you prefer no holdout
    val_frac=0.1,
    random_state=16,
)

# 3) Save model + metadata (threshold + feature ordering)
model_path, meta_path = save_model_bundle(
    model=final_model,
    threshold_abs=best_threshold,
    feature_names=list(X_all.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/error_line/",
    name="drop_na_cols_50_recent_data_xgb_line_error_04_03_26",
)

print("Saved:", model_path)
print("Saved:", meta_path)

# 4) Load later
loaded_model, meta = load_model_bundle(model_path, meta_path)
print("Loaded threshold_abs:", meta["threshold_abs"])


Saved: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/error_line/drop_na_cols_50_recent_data_xgb_line_error_04_03_26.joblib
Saved: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/error_line/drop_na_cols_50_recent_data_xgb_line_error_04_03_26.meta.json
Loaded threshold_abs: 0.5
